# Core 모듈 테스트

리팩토링 후 `core/` (Pinocchio_Wrapper, Mujoco_Kernel, FloatingBaseRobotState)의 동작 검증.

검증 항목:
1. Pinocchio_Wrapper — instantiation, update_model, properties (상태/동역학/기구학/중심운동량), get_* 메서드, 좌표 변환
2. Mujoco_Kernel — init, reset, state I/O, ctrl_tau, step, contact
3. End-to-end — kernel ↔ state ↔ wrapper 동기화

기대 결과: 모든 cell이 assertion 없이 통과.

In [ ]:
import sys
from pathlib import Path

_HERE = Path.cwd() if '__file__' not in globals() else Path(__file__).resolve().parent
_ROOT = _HERE.parents[0]
sys.path.insert(0, str(_ROOT / 'mj_sim' / 'humanoid'))

import numpy as np
import pinocchio as pin

from core import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel
from core.pinocchio_wrapper import G1_FRAMES, LEG_KEYS, LEG_IDX

%load_ext autoreload
%autoreload 2

URDF = str(_ROOT / 'models' / 'g1' / 'urdf' / 'g1_29dof.urdf')
PKG  = str(_ROOT / 'models' / 'g1')
MJCF = str(_ROOT / 'models' / 'scenes' / 'g1' / 'scene_29dof.xml')

print(f'pinocchio: {pin.__version__}')
print(f'LEG_KEYS : {LEG_KEYS}')
print(f'LEG_IDX  : {LEG_IDX}')

## 1. Pinocchio_Wrapper

In [ ]:
wrapper = Pinocchio_Wrapper(URDF, PKG)

print(f'nv  = {wrapper.nv}   (expected 35 for G1 29dof + free flyer 6)')
print(f'na  = {wrapper.na}   (expected 29)')
print(f'mass = {wrapper.mass:.3f} kg')
print(f'leg_keys = {wrapper.leg_keys}')
print(f'frame ids: {list(wrapper.fid.keys())}')
print(f'hip_placements keys: {list(wrapper.hip_placements.keys())}')

assert wrapper.nv == 35
assert wrapper.na == 29
assert set(wrapper.hip_placements.keys()) == {'L_foot', 'R_foot'}
assert not hasattr(wrapper, 'state_curr'), 'state_curr 제거 안 됨'
print('\n✅ instantiation OK')

### 1.1 R_z 버퍼 동작 (이전 @property 충돌 회귀 확인)

In [ ]:
# neutral pose에서 R_z = I 인지
q = wrapper.q_init.copy()
dq = np.zeros(wrapper.nv)
wrapper.update_model(q, dq)

assert np.allclose(wrapper.R_z, np.eye(3)), f'neutral에서 R_z != I:\n{wrapper.R_z}'

# yaw 90도 회전 후 R_z 갱신 확인
q_yaw = q.copy()
yaw = np.pi / 2
q_yaw[3:7] = [0, 0, np.sin(yaw/2), np.cos(yaw/2)]   # quat (x,y,z,w)
wrapper.update_model(q_yaw, dq)

expected_R_z = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]])
assert np.allclose(wrapper.R_z, expected_R_z, atol=1e-6), f'R_z mismatch:\n{wrapper.R_z}'
print('R_z(yaw=90°) =\n', wrapper.R_z)
print('\n✅ R_z in-place 갱신 OK')

# 원상복귀
wrapper.update_model(wrapper.q_init.copy(), dq)

### 1.2 동역학 property

In [ ]:
M     = wrapper.M
M_inv = wrapper.M_inv
C     = wrapper.C
nle   = wrapper.nle
g     = wrapper.g

print(f'M      shape = {M.shape}')
print(f'M_inv  shape = {M_inv.shape}')
print(f'C      shape = {C.shape}')
print(f'nle    shape = {nle.shape}')
print(f'g      shape = {g.shape}')

# M, M_inv shape
assert M.shape == (wrapper.nv, wrapper.nv)
assert M_inv.shape == (wrapper.nv, wrapper.nv)

# M @ M_inv ≈ I (수치 유효성)
I_est = M @ M_inv
err = np.linalg.norm(I_est - np.eye(wrapper.nv))
print(f'\n||M @ M_inv - I|| = {err:.2e}')
assert err < 1e-6, 'M_inv 수치 부정확'

# M 대칭성
assert np.allclose(M, M.T, atol=1e-9), 'M 비대칭'

# nle, g 형태
assert nle.shape == (wrapper.nv,)
assert g.shape == (wrapper.nv,)

print('\n✅ 동역학 property OK')

### 1.3 기구학 property

In [ ]:
print(f'base_pos      = {wrapper.base_pos}')
print(f'com_pos_world = {wrapper.com_pos_world}')
print(f'com_vel_world = {wrapper.com_vel_world}  (neutral+0vel → 모두 0)')
print(f'J_com  shape  = {wrapper.J_com.shape}')

assert wrapper.base_pos.shape == (3,)
assert wrapper.com_pos_world.shape == (3,)
assert wrapper.com_vel_world.shape == (3,)
assert wrapper.J_com.shape == (3, wrapper.nv)
assert np.allclose(wrapper.com_vel_world, 0, atol=1e-9), 'dq=0 인데 vcom != 0'

print('\n✅ 기구학 property OK')

### 1.4 중심 운동량 property

In [ ]:
print(f'hg            = {np.asarray(wrapper.hg)}')
print(f'Ag      shape = {wrapper.Ag.shape}')
print(f'angular_momentum = {wrapper.angular_momentum}')
print(f'inertia_matrix shape = {wrapper.inertia_matrix.shape}')

assert wrapper.Ag.shape == (6, wrapper.nv)
assert wrapper.inertia_matrix.shape == (3, 3)

# inertia_matrix 대칭 + positive definite
Ig = wrapper.inertia_matrix
assert np.allclose(Ig, Ig.T, atol=1e-9), 'Ig 비대칭'
eigs = np.linalg.eigvalsh(Ig)
assert (eigs > 0).all(), f'Ig가 positive definite 아님: eigs={eigs}'
print(f'Ig eigenvalues: {eigs}')

print('\n✅ 중심 운동량 property OK')

### 1.5 Jacobian get_*

In [ ]:
for key in ['L_foot', 'R_foot', 'L_hand', 'R_hand']:
    Jw = wrapper.get_J_world(key)
    Jb = wrapper.get_J_body(key)
    Jd = wrapper.get_Jdot_dq_world(key)
    
    assert Jw.shape == (6, wrapper.nv), f'{key} J_world shape {Jw.shape}'
    assert Jb.shape == (6, wrapper.nv), f'{key} J_body shape {Jb.shape}'
    assert Jd.shape == (6,),            f'{key} Jdot_dq shape {Jd.shape}'
    
    # dq=0 → Jdot @ dq = 0
    assert np.allclose(Jd, 0, atol=1e-9), f'{key} Jdot_dq != 0 with dq=0'

print('✅ get_J_world / get_J_body / get_Jdot_dq_world OK')

### 1.6 프레임 조회 + 좌표 변환

In [ ]:
# get_ee_state_world
oMf, lin, ang = wrapper.get_ee_state_world('L_foot')
print(f'L_foot translation: {oMf.translation}')
print(f'L_foot lin/ang vel: {lin}, {ang}')
assert lin.shape == (3,) and ang.shape == (3,)
assert np.allclose(lin, 0) and np.allclose(ang, 0), 'dq=0인데 ee twist != 0'

# get_moment_arm_in_world (L_foot - com)
ma = wrapper.get_moment_arm_in_world('L_foot')
expected = wrapper.oM_Lfoot.translation - wrapper.com_pos_world
assert np.allclose(ma, expected)
print(f'L_foot moment arm (foot - com): {ma}')

# get_hip_offset
for k in ['L_foot', 'R_foot']:
    off = wrapper.get_hip_offset(k)
    print(f'{k} hip offset (base frame): {off}')
    assert off.shape == (3,)

# 좌표 변환 round-trip
p_world = np.array([0.5, -0.2, 0.8])
p_base  = wrapper.world_to_base_frame(p_world)
# inverse: base->world = oMb * p_base
p_back  = wrapper.oMb.translation + wrapper.R_body_to_world @ p_base
assert np.allclose(p_back, p_world, atol=1e-9), 'world↔base round-trip 실패'
print(f'\nworld→base→world round-trip err: {np.linalg.norm(p_back - p_world):.2e}')

print('\n✅ 프레임 조회 + 좌표 변환 OK')

## 2. FloatingBaseRobotState (standalone)

In [ ]:
state = FloatingBaseRobotState()

assert state.NQ == 36 and state.NV == 35
assert state.q.shape  == (36,)
assert state.dq.shape == (35,)

# partition view 확인
print(f'left_leg_angle  shape = {state.left_leg_angle.shape}  (6)')
print(f'right_arm_angle shape = {state.right_arm_angle.shape}  (7)')

# setter (np.copyto) 동작
q_new = np.linspace(0, 1, 36)
state.q = q_new
assert np.allclose(state.q, q_new)

# partition view가 살아있는지
assert np.allclose(state.left_leg_angle, q_new[7:13])

print('\n✅ FloatingBaseRobotState OK')

## 3. Mujoco_Kernel

In [ ]:
joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]
print(f'Pinocchio actuated joints (na={wrapper.na}, names={len(joint_names)}):')
for n in joint_names[:5]:
    print(f'  - {n}')
print('  ...')

kernel = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names)

print(f'\nmodel.nq = {kernel.model.nq}, nv = {kernel.model.nv}, nu = {kernel.model.nu}')
print(f'q_keyframe present: {kernel.q_keyframe is not None}')
if kernel.q_keyframe is not None:
    print(f'q_keyframe shape: {kernel.q_keyframe.shape}')

print('\n✅ kernel instantiation OK')

### 3.1 reset + state read/write

In [ ]:
kernel.reset_to_keyframe(0)

# read
print(f'q_mj  shape = {kernel.q_mj.shape}')
print(f'dq_mj shape = {kernel.dq_mj.shape}')
print(f'q_mj[:7] (base 7) = {kernel.q_mj[:7]}')

# state로 변환
kernel.update_robot_state(state)
print(f'\nstate.q[:7]  (Pin 순서, base) = {state.q[:7]}')
print(f'state.q[3:7] (Pin quat x,y,z,w) = {state.q[3:7]}')
print(f'MJ quat[3:7] (w,x,y,z)        = {kernel.q_mj[3:7]}')

# quat 순서 변환 검증: MJ(w,x,y,z) → Pin(x,y,z,w)
assert np.isclose(state.q[3], kernel.q_mj[4]), 'quat x mismatch'
assert np.isclose(state.q[4], kernel.q_mj[5]), 'quat y mismatch'
assert np.isclose(state.q[5], kernel.q_mj[6]), 'quat z mismatch'
assert np.isclose(state.q[6], kernel.q_mj[3]), 'quat w mismatch'

# ctrl_tau write
tau = np.zeros(wrapper.na)
kernel.ctrl_tau = tau
assert np.allclose(kernel.data.ctrl, 0)

print('\n✅ reset / state I/O / quat 변환 OK')

### 3.2 step + 접촉 등록

In [ ]:
kernel.register_foot_geoms({
    'L_foot': 'left_ankle_roll_link',
    'R_foot': 'right_ankle_roll_link',
})

# 1 step 진행
kernel.step()
kernel.update_robot_state(state)

# 접촉 조회
forces, points = kernel.get_foot_contact_state()
print(f'foot keys: {list(forces.keys())}')
for k, f in forces.items():
    print(f'  {k}: f_world = {f}, |f| = {np.linalg.norm(f):.2f} N')
print(f'contact points: {len(points)}')

print('\n✅ step + 접촉 OK')

## 4. End-to-end 동기화

kernel → state → wrapper로 q/dq가 일관되게 전달되는지.

In [ ]:
kernel.reset_to_keyframe(0)

# 1) MJ → state
kernel.update_robot_state(state)

# 2) state → wrapper
wrapper.update_model(state.q, state.dq)

# 3) wrapper의 frame placements가 state.q로부터 유도된 값인지
#    (간접 검증: base 위치가 state의 humanoid_base_pos와 일치)
assert np.allclose(wrapper.base_pos, state.humanoid_base_pos), \
    f'base_pos mismatch: wrapper={wrapper.base_pos} state={state.humanoid_base_pos}'

print(f'wrapper.base_pos      = {wrapper.base_pos}')
print(f'state.humanoid_base_pos = {state.humanoid_base_pos}')
print(f'wrapper.com_pos_world = {wrapper.com_pos_world}')
print(f'wrapper.oM_Lfoot.t    = {wrapper.oM_Lfoot.translation}')
print(f'wrapper.oM_Rfoot.t    = {wrapper.oM_Rfoot.translation}')

# CoM이 두 발 사이 어딘가에 있어야
lf_z = wrapper.oM_Lfoot.translation[2]
rf_z = wrapper.oM_Rfoot.translation[2]
com_z = wrapper.com_pos_world[2]
assert com_z > max(lf_z, rf_z), f'CoM 높이가 발보다 낮음: com={com_z} feet={lf_z}/{rf_z}'

print(f'\nCoM 높이 = {com_z:.3f} m (발 높이 max = {max(lf_z, rf_z):.3f} m)')
print('\n✅ end-to-end 동기화 OK')

## 5. update_model 다회 호출 시 캐시 일관성

In [ ]:
# 같은 q/dq로 두 번 호출 → 결과 동일해야 함
wrapper.update_model(state.q, state.dq)
M1     = wrapper.M.copy()
M_inv1 = wrapper.M_inv.copy()
Jcom1  = wrapper.J_com.copy()
Jw1    = wrapper.get_J_world('L_foot').copy()

wrapper.update_model(state.q, state.dq)
M2     = wrapper.M
M_inv2 = wrapper.M_inv
Jcom2  = wrapper.J_com
Jw2    = wrapper.get_J_world('L_foot')

assert np.allclose(M1, M2)
assert np.allclose(M_inv1, M_inv2)
assert np.allclose(Jcom1, Jcom2)
assert np.allclose(Jw1, Jw2)

print('✅ update_model 멱등성 (same q,dq → same outputs) OK')

## 6. yaw unwrap (FloatingBaseRobotState.compute_rpy_world)

In [ ]:
# yaw가 +178° → -179° 점프 시퀀스로 unwrap 검증
s = FloatingBaseRobotState()

def set_yaw(yaw):
    qx, qy, qz, qw = 0, 0, np.sin(yaw/2), np.cos(yaw/2)
    s._q[3:7] = [qx, qy, qz, qw]

yaw_seq = np.deg2rad([175, 178, -179, -176, -170, 175, 170])
yaw_unwrapped = []
for y in yaw_seq:
    set_yaw(y)
    yaw_unwrapped.append(s.compute_rpy_world()[2])

yaw_unwrapped = np.array(yaw_unwrapped)
deltas = np.diff(yaw_unwrapped)

print('raw yaw input  (deg):', np.rad2deg(yaw_seq).round(1))
print('unwrapped yaw  (deg):', np.rad2deg(yaw_unwrapped).round(1))
print('step deltas    (deg):', np.rad2deg(deltas).round(1))

# 모든 step delta가 (-π, π) 안 — 360° 점프 없음
assert (np.abs(deltas) < np.pi).all(), f'unwrap 실패: deltas(rad)={deltas}'

# reset 후 첫 호출이 raw quat yaw 그대로 나오는지
s.reset_yaw_unwrap()
set_yaw(np.deg2rad(170))
rpy = s.compute_rpy_world()
assert np.isclose(rpy[2], np.deg2rad(170), atol=1e-6), \
    f'reset 후 첫 호출이 raw yaw와 다름: {np.rad2deg(rpy[2])}'

print('\n✅ yaw unwrap + reset OK')

## 종합 결과
위 cell이 전부 통과했다면 core 리팩토링은 회귀 없음.